**CELL 1 — Imports**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
import json


**CELL 2 — Define Paths**

In [0]:
BASE_PATH = "abfss://telecom-project@projectstorageaccvijay.dfs.core.windows.net"

paths = {
    "bronze": f"{BASE_PATH}/bronze",
    "silver": f"{BASE_PATH}/silver",
    "error":  f"{BASE_PATH}/errors",
    "audit":  f"{BASE_PATH}/audit",
    "config": f"{BASE_PATH}/config"
}

PIPELINE_STEP = "bronze-to-silver"


**CELL 2.1 — Table → Bronze File Mapping**

In [0]:
TABLE_FILE_MAPPING = {
    "fact_customer": "dbo.fact_customer.csv",
    "dim_cdr": "dbo.dim_cdr.csv",
    "dim_recharge": "dbo.dim_recharge.csv",
    "dim_internet_usage": "dbo.dim_internet_usage.csv",
    "dim_complaint": "dbo.dim_complaint.csv",
    "dim_tower": "dbo.dim_tower.csv"
}


**CELL 3 — Load schema_validation.json**

In [0]:
SCHEMA_FILE = f"{paths['config']}/schema_validation.json"

schema_df = spark.read.text(SCHEMA_FILE)
schema_str = "\n".join(r.value for r in schema_df.collect())
schema_def = json.loads(schema_str)

schema_def.keys()


**CELL 4 — Create Audit Delta Table (Append-only)**

In [0]:
audit_path = f"{paths['audit']}/audit_log"

audit_schema = StructType([
    StructField("table_name", StringType()),
    StructField("step_name", StringType()),
    StructField("function_name", StringType()),
    StructField("file_path", StringType()),
    StructField("status", StringType()),
    StructField("event_ts", TimestampType()),
    StructField("error_message", StringType())
])

spark.createDataFrame([], audit_schema) \
    .write.format("delta") \
    .mode("ignore") \
    .save(audit_path)


**CELL 5 — Audit Logger Function**

In [0]:
def now_ts():
    return datetime.now(timezone.utc)

def write_audit(table, step, fn, path, status, error=None):
    row = [(table, step, fn, path, status, now_ts(), error)]
    spark.createDataFrame(row, audit_schema) \
        .write.format("delta") \
        .mode("append") \
        .save(audit_path)


**CELL 6 — Read Bronze CSV**

In [0]:
def read_bronze(table):
    file_name = TABLE_FILE_MAPPING.get(table)

    if not file_name:
        raise Exception(f"No bronze file mapping found for table {table}")

    path = f"{paths['bronze']}/{file_name}"

    write_audit(table, PIPELINE_STEP, "read_bronze", path, "START")

    df = (
        spark.read
             .option("header", True)
             .option("inferSchema", True)
             .csv(path)
    )

    write_audit(table, PIPELINE_STEP, "read_bronze", path, "SUCCESS")
    return df


**CELL 7 — Enforce Schema (same logic)**

In [0]:
def enforce_schema(table, df):
    write_audit(table, PIPELINE_STEP, "enforce_schema", table, "START")

    for col, dtype in schema_def[table]["columns"].items():
        if dtype.lower() == "timestamp":
            df = df.withColumn(col, F.to_timestamp(col))
        else:
            df = df.withColumn(col, F.col(col).cast(dtype))

    write_audit(table, PIPELINE_STEP, "enforce_schema", table, "SUCCESS")
    return df


**CELL 8 — Column Validation**

In [0]:
def column_validation(df, table):
    path = f"{paths['bronze']}/{table}.csv"

    expected = set(schema_def[table]["columns"].keys())
    actual = set(df.columns)

    extra_cols = list(actual - expected)
    missing_cols = list(expected - actual)

    if missing_cols:
        write_audit(
            table, PIPELINE_STEP, "column_validation",
            path, "FAILED", f"Missing columns: {missing_cols}"
        )
        raise Exception(f"Missing columns: {missing_cols}")

    if extra_cols:
        err_path = f"{paths['error']}/{table}/extra_columns"
        df.select(*extra_cols) \
            .write.mode("overwrite") \
            .option("header", True) \
            .csv(err_path)

        df = df.drop(*extra_cols)

        write_audit(
            table, PIPELINE_STEP, "column_validation",
            path, "SUCCESS", f"Extra columns removed: {extra_cols}"
        )
    else:
        write_audit(
            table, PIPELINE_STEP, "column_validation",
            path, "SUCCESS", "No column mismatch"
        )

    return df


**CELL 9 — Null Check (exact counts)**

In [0]:
def null_check(df, table):
    path = f"{paths['bronze']}/{table}.csv"
    not_null_cols = schema_def[table].get("not_null_cols", [])
    total_nulls = 0

    for col in not_null_cols:
        bad = df.filter(F.col(col).isNull())
        cnt = bad.count()

        if cnt > 0:
            total_nulls += cnt
            err_path = f"{paths['error']}/{table}/null_records"
            bad.write.mode("append").option("header", True).csv(err_path)
            df = df.filter(F.col(col).isNotNull())

    write_audit(
        table, PIPELINE_STEP, "null_check",
        path, "SUCCESS", f"null_record_count={total_nulls}"
    )

    return df


**CELL 10 — Deduplication**

In [0]:
def deduplicate(df, table):
    path = f"{paths['bronze']}/{table}.csv"
    pk = schema_def[table].get("primary_key", [])

    if not pk:
        write_audit(
            table, PIPELINE_STEP, "deduplicate",
            path, "SUCCESS", "No PK"
        )
        return df

    w = Window.partitionBy(*pk).orderBy(F.lit(1))
    df_rn = df.withColumn("_rn", F.row_number().over(w))

    duplicates = df_rn.filter(F.col("_rn") > 1)
    dup_count = duplicates.count()

    if dup_count > 0:
        err_path = f"{paths['error']}/{table}/duplicate_records"
        duplicates.write.mode("overwrite").option("header", True).csv(err_path)

    df = df_rn.filter(F.col("_rn") == 1).drop("_rn")

    write_audit(
        table, PIPELINE_STEP, "deduplicate",
        path, "SUCCESS", f"duplicate_record_count={dup_count}"
    )

    return df


**CELL 11 — Write Silver**

In [0]:
def write_silver(df, table):
    silver_path = f"{paths['silver']}/{table}"

    df.write.format("delta") \
        .mode("overwrite") \
        .save(silver_path)


**CELL 12 — process_table()**

In [0]:
def process_table(table):
    try:
        write_audit(table, "PIPELINE", "process_table", table, "START")

        df = read_bronze(table)
        df = enforce_schema(table, df)
        df = column_validation(df, table)
        df = null_check(df, table)
        df = deduplicate(df, table)
        write_silver(df, table)

        write_audit(table, "PIPELINE", "process_table", table, "SUCCESS")
        return {"table": table, "status": "SUCCESS"}

    except Exception as e:
        write_audit(
            table, "PIPELINE", "process_table",
            table, "FAILED", str(e)
        )
        return {"table": table, "status": "FAILED", "error": str(e)}


**CELL 13 — Parallel Execution**

In [0]:
def run_parallel(max_workers=5):
    tables = list(schema_def.keys())
    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_table, t) for t in tables]

        for f in as_completed(futures):
            results.append(f.result())

    return results


**CELL 14 — RUN PIPELINE**

In [0]:
run_parallel(max_workers=5)


In [0]:
# dbutils.fs.rm(
#     "abfss://telecom-project@projectstorageaccvijay.dfs.core.windows.net/audit/audit_log",
#     recurse=True
# )


**CELL 15 — View Audit Log**

In [0]:
spark.read.format("delta") \
    .load(audit_path) \
    .orderBy(F.col("event_ts").desc()) \
    .display()
